# Module X2: Survival analysis (time-to-event data)

**Goal:** analyse **how long until something happens** (a customer churns, a patient relapses, a machine fails, a student drops out) when some units haven't had the event **yet** by the time observation ends. That's **censoring**, and it breaks ordinary methods.

**You will learn:** censoring and why naive averages and yes/no models fail · the survival function, hazard and cumulative hazard · the **Kaplan–Meier** estimator and its confidence interval *by hand* ·
comparing groups with the **log-rank test** · the exponential model with censoring · **Cox proportional-hazards regression** (including the partial likelihood by hand) · checking proportional hazards · reporting

**Maths in this module:** $S(t)$, $h(t)$, $H(t)$ and $S(t)=e^{-H(t)}$ · the product-limit estimator $\hat S(t)=\prod(1-d_i/n_i)$ · Greenwood's variance · log-rank statistic · exponential MLE with censoring $\hat\lambda=\text{events}/\text{total time}$ · the Cox partial likelihood · hazard ratios

> 🔗 **Prerequisites:** Modules 7b (likelihood), 8b (tests, CIs) and 9 (regression). Uses only `statsmodels`, no extra packages.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize_scalar
from statsmodels.duration.hazard_regression import PHReg
from statsmodels.duration.survfunc import SurvfuncRight, survdiff

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", palette="colorblind")
churn = pd.read_csv(DATA / "customer_churn.csv")
churn.head()

## The data

`customer_churn.csv`: 400 subscribers followed from sign-up. For each: `plan` (basic or premium), `monthly_spend`, `tenure_months` (time until they **churned, or until follow-up ended**) and `churned` (**1 = churned, 0 = censored**, still a customer when observation stopped).
**Question:** do basic-plan customers churn faster than premium customers, and by how much?

In [ ]:
print(f"{len(churn)} customers; {churn['churned'].sum()} churned, {(churn['churned'] == 0).sum()} censored ({(churn['churned'] == 0).mean():.1%}); longest follow-up {churn['tenure_months'].max():.0f} months")

sample = churn.sample(25, random_state=1).sort_values("tenure_months").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, r in sample.iterrows():
    ax.hlines(i, 0, r["tenure_months"], color="grey", linewidth=1.5)
    ax.plot(r["tenure_months"], i, "x" if r["churned"] == 1 else "o", color="tab:red" if r["churned"] == 1 else "tab:blue", markersize=7)
ax.set_xlabel("Months since sign-up"); ax.set_ylabel("customer (25 shown)"); ax.set_yticks([])
ax.set_title("red × = churned (event observed)    blue ○ = censored (still a customer)")
plt.show()

## 1. Why censoring breaks ordinary methods

A blue circle means: *we only know this customer lasted **at least** that long.* Three tempting but wrong analyses:

In [ ]:
t, e = churn["tenure_months"].to_numpy(), churn["churned"].to_numpy()
print(f"mean of all observed times (treats censored times as if they were churn times): {t.mean():.2f} months  -> too low")
print(f"mean among churned customers only (throws away the loyal, censored ones)      : {t[e == 1].mean():.2f} months  -> too low")
print("A yes/no model (\"did they churn?\") ignores WHEN, and who was observed long enough to churn.")

* **Averaging all times** counts a customer who is *still loyal at month 20* as if they left at month 20.
* **Averaging only churners** discards exactly the long-surviving customers, biasing the average down.
* **Logistic regression on `churned`** treats a customer followed for 2 months like one followed for 30, and ignores timing.

Survival analysis uses everything each customer tells us: *they were still here at time $t_i$ and then (if $\delta_i=1$) left.*

## 2. The core quantities

Let $T$ be the time to the event.

| quantity | definition | meaning |
|---|---|---|
| **survival function** $S(t)$ | $P(T>t)$ | proportion still "alive" (here: still customers) at $t$ |
| **hazard** $h(t)$ | $\lim\limits_{\Delta\to0}\dfrac{P(t\le T<t+\Delta\mid T\ge t)}{\Delta}$ | instantaneous churn rate among those still here |
| **cumulative hazard** $H(t)$ | $\int_0^th(u)\,du$ | accumulated risk; $S(t)=e^{-H(t)}$ |

With a constant hazard $\lambda$ (exponential model): $S(t)=e^{-\lambda t}$, mean $1/\lambda$, median $\ln2/\lambda$.

## 3. Kaplan–Meier: the survival curve, by hand

At each distinct event time $t_i$, let $n_i$ = customers **at risk** (still under observation just before $t_i$) and $d_i$ = churn events at $t_i$. The chance of surviving past $t_i$, given you reached it, is $1-d_i/n_i$. Multiply along the way:

$$\hat S(t)=\prod_{t_i\le t}\Big(1-\frac{d_i}{n_i}\Big)\qquad\text{Greenwood: }\ \widehat{\operatorname{Var}}\big[\hat S(t)\big]=\hat S(t)^2\sum_{t_i\le t}\frac{d_i}{n_i(n_i-d_i)}$$

Censored customers stay in the denominator $n_i$ until the moment they drop out of observation, then contribute no more.

In [ ]:
def kaplan_meier(time, event):
    time, event = np.asarray(time, dtype=float), np.asarray(event)
    rows, S, gw = [], 1.0, 0.0
    for u in np.unique(time[event == 1]):
        n_i = int((time >= u).sum())
        d_i = int(((time == u) & (event == 1)).sum())
        S *= 1 - d_i / n_i
        gw += d_i / (n_i * (n_i - d_i)) if n_i > d_i else np.inf
        rows.append((u, n_i, d_i, S, S * math.sqrt(gw), gw))
    return pd.DataFrame(rows, columns=["t", "n_risk", "d", "S", "se", "greenwood_sum"])

km = kaplan_meier(t, e)
print(km.drop(columns='greenwood_sum').head(4).round(4).to_string(index=False))

ref = SurvfuncRight(t, e)
print("\nmatches statsmodels (survival and Greenwood SE):", np.allclose(km["S"], ref.surv_prob), np.allclose(km["se"], ref.surv_prob_se))

In [ ]:
def survival_at(km, time):
    before = km[km["t"] <= time]
    return 1.0 if before.empty else before["S"].iloc[-1]

def median_survival(km):
    below = km[km["S"] <= 0.5]
    return below["t"].iloc[0] if len(below) else np.nan

S12 = survival_at(km, 12)
row = km[km["t"] <= 12].iloc[-1]
se_ll = math.sqrt(row["greenwood_sum"]) / abs(math.log(S12))                    # SE of log(-log S)
ci_plain = (S12 - 1.96 * row["se"], S12 + 1.96 * row["se"])
ci_loglog = (S12 ** math.exp(1.96 * se_ll), S12 ** math.exp(-1.96 * se_ll))     # always stays inside (0, 1)
print(f"S(12 months) = {S12:.3f}; plain 95% CI ({ci_plain[0]:.3f}, {ci_plain[1]:.3f}); log-log 95% CI ({ci_loglog[0]:.3f}, {ci_loglog[1]:.3f})")
print(f"median survival time (first time S(t) <= 0.5): {median_survival(km):.2f} months")
print("at risk at 0, 6, 12, 18, 24 months:", [int((t >= m).sum()) for m in (0, 6, 12, 18, 24)])

About **44.5%** of customers survive past 12 months (95% CI 39.6% to 49.3%), and the **median** customer lasts about **10.1 months**. The *at-risk counts* fall from 400 to 34 by month 24, so the right-hand end of the curve rests on few people: its confidence interval is wide.
The **log-log** interval is preferred in practice because it can never leave $[0,1]$ (the two agree here because $\hat S$ is mid-range).

## 4. Comparing groups: Kaplan–Meier by plan and the log-rank test

In [ ]:
def km_plot(ax, df, label, color):
    k = kaplan_meier(df["tenure_months"], df["churned"])
    xs = np.r_[0, k["t"]]; ys = np.r_[1, k["S"]]; se = np.r_[0, k["se"]]
    ax.step(xs, ys, where="post", color=color, label=label)
    ax.fill_between(xs, np.clip(ys - 1.96 * se, 0, 1), np.clip(ys + 1.96 * se, 0, 1), step="post", color=color, alpha=0.15)
    cens = df[df["churned"] == 0]["tenure_months"].to_numpy()
    ax.plot(cens, [survival_at(k, c) for c in cens], "|", color=color, markersize=8)
    return k

fig, ax = plt.subplots(figsize=(7.5, 4.2))
km_by = {plan: km_plot(ax, d, plan, c) for (plan, d), c in zip(churn.groupby("plan"), ("tab:red", "tab:blue"))}
ax.axhline(0.5, color="grey", linestyle=":"); ax.set_xlabel("Months since sign-up"); ax.set_ylabel("Proportion still a customer")
ax.set_title("Premium customers stay longer (| = censored)"); ax.legend(); plt.show()

for plan, k in km_by.items():
    print(f"{plan:8s} median {median_survival(k):5.2f} months | S(12) = {survival_at(k, 12):.3f} | S(24) = {survival_at(k, 24):.3f}")

**Log-rank test** ($H_0$: the two groups have identical survival curves). At every event time, compare the churners observed in group 1 with the number *expected* if group membership didn't matter (the group's share of those at risk):

$$E_1=\sum_i d_i\frac{n_{1i}}{n_i},\qquad V=\sum_i d_i\frac{n_{1i}}{n_i}\Big(1-\frac{n_{1i}}{n_i}\Big)\frac{n_i-d_i}{n_i-1},\qquad \chi^2=\frac{(O_1-E_1)^2}{V}\sim\chi^2_1$$

In [ ]:
def logrank(time, event, group):
    time, event, group = np.asarray(time, dtype=float), np.asarray(event), np.asarray(group)
    g1 = group == np.unique(group)[0]
    O = E = V = 0.0
    for u in np.unique(time[event == 1]):
        at_risk = time >= u
        n, n1 = at_risk.sum(), (at_risk & g1).sum()
        d, d1 = ((time == u) & (event == 1)).sum(), ((time == u) & (event == 1) & g1).sum()
        O += d1
        E += d * n1 / n
        V += d * (n1 / n) * (1 - n1 / n) * (n - d) / (n - 1) if n > 1 else 0
    chi2 = (O - E) ** 2 / V
    return chi2, stats.chi2.sf(chi2, 1), O, E

chi2, p, O, E = logrank(t, e, churn["plan"])
print(f"first group (basic): observed churners {O:.0f}, expected under 'no difference' {E:.1f}")
print(f"log-rank chi2 = {chi2:.2f}, p = {p:.1e}      statsmodels: {survdiff(t, e, churn['plan'].to_numpy())}")

Basic-plan customers churned far more than expected (201 observed vs 155 expected), with $p\approx9\times10^{-8}$. The log-rank test tells you *whether* the curves differ, not *how much*; for size, use a hazard ratio (§6) or a median/restricted-mean difference.

## 5. The exponential model with censoring (a likelihood you can do by hand)

If the hazard is constant, $\lambda$, a churner at time $t_i$ contributes the density $\lambda e^{-\lambda t_i}$ and a **censored** customer contributes the survival probability $e^{-\lambda t_i}$. The log-likelihood is
$\ell(\lambda)=d\ln\lambda-\lambda\sum t_i$ (with $d$ = number of events), so
$$\hat\lambda=\frac{d}{\sum_it_i}=\frac{\text{events}}{\text{total time at risk}},\qquad \text{SE}\big(\ln\hat\lambda\big)\approx\frac1{\sqrt d}.$$
Note that censored customers add their **time** to the denominator but no event to the numerator: that is how they are used without bias.

In [ ]:
def exponential_rate(time, event):
    return np.sum(event) / np.sum(time)

rates = {}
for plan, d in churn.groupby("plan"):
    lam = exponential_rate(d["tenure_months"], d["churned"])
    half = 1.96 / math.sqrt(d["churned"].sum())
    rates[plan] = lam
    print(f"{plan:8s} {d['churned'].sum():3d} events / {d['tenure_months'].sum():7.1f} customer-months = {lam:.4f} per month "
          f"(95% CI {lam * math.exp(-half):.4f} to {lam * math.exp(half):.4f}); implied median {math.log(2) / lam:.1f} months; S(12) = {math.exp(-lam * 12):.3f}")
print(f"\nrate ratio basic / premium = {rates['basic'] / rates['premium']:.2f}")

Basic customers churn at about **0.089 per month** versus **0.046** for premium: a **rate ratio of 1.92**. The model-based medians (7.8 and 15.0 months) and $S(12)$ (0.344 for basic) are close to the Kaplan–Meier values (7.8, 14.5; 0.362): the constant-hazard assumption is reasonable here. In real data, compare the model curve with KM before trusting it.

## 6. Cox proportional-hazards regression

Cox regression models the **hazard** as a baseline times an exponential of the covariates, leaving the baseline shape $h_0(t)$ **unspecified**:

$$h(t\mid x)=h_0(t)\,e^{\beta^\top x}\qquad\Longrightarrow\qquad \text{HR}=e^{\beta_j}\ \text{ for a one-unit increase in }x_j$$

A **hazard ratio** of 1.9 means: at any moment, a basic customer has 1.9 times the instantaneous churn risk of a premium customer with the same spend. The $\beta$'s are found by maximising the **partial likelihood**: at each event time, the probability that the person who churned is *that* person among everyone at risk:

$$L(\beta)=\prod_{i:\ \delta_i=1}\frac{e^{\beta^\top x_i}}{\sum_{j\in R(t_i)}e^{\beta^\top x_j}}$$

where $R(t_i)$ is the risk set at $t_i$. Here it is by hand for one binary covariate (basic = 1):

In [ ]:
basic = (churn["plan"] == "basic").to_numpy().astype(float)

def neg_partial_loglik(beta):
    ll = 0.0
    for i in np.where(e == 1)[0]:
        risk_set = t >= t[i]
        ll += beta * basic[i] - np.log(np.exp(beta * basic[risk_set]).sum())
    return -ll

fit = minimize_scalar(neg_partial_loglik, bounds=(-3, 3), method="bounded")
h = 1e-4
se = 1 / math.sqrt((neg_partial_loglik(fit.x + h) - 2 * neg_partial_loglik(fit.x) + neg_partial_loglik(fit.x - h)) / h ** 2)    # curvature of the log-likelihood
print(f"by hand: beta = {fit.x:.4f}, SE = {se:.4f}  ->  HR = {math.exp(fit.x):.2f} (95% CI {math.exp(fit.x - 1.96 * se):.2f} to {math.exp(fit.x + 1.96 * se):.2f})")

cox_simple = PHReg.from_formula("tenure_months ~ basic", churn.assign(basic=basic), status=e).fit()
print(f"statsmodels: beta = {cox_simple.params[0]:.4f}, SE = {cox_simple.bse[0]:.4f}")

The hand-built partial likelihood reproduces the library result. Now add the other covariate:

In [ ]:
data = churn.assign(basic=basic)
cox = PHReg.from_formula("tenure_months ~ basic + monthly_spend", data, status=e).fit()
ci = cox.conf_int()
table = pd.DataFrame({"HR": np.exp(cox.params), "ci_low": np.exp(ci[:, 0]), "ci_high": np.exp(ci[:, 1]), "p": cox.pvalues}, index=cox.model.exog_names).round(4)
print(table)
b_sp = cox.params[1]
print(f"\nper $10 more monthly spend: HR = {math.exp(10 * b_sp):.2f} (95% CI {math.exp(10 * ci[1, 0]):.2f} to {math.exp(10 * ci[1, 1]):.2f})")

**Interpretation:** holding spend fixed, basic-plan customers have **1.9 times the churn hazard** of premium customers (95% CI 1.50 to 2.42). Holding plan fixed, each extra **\$10** of monthly spend is associated with a **15% lower** hazard (HR 0.85, 95% CI 0.78 to 0.94).
The hazard ratio is *relative*: it doesn't tell you the absolute risk. Pair it with $S(12)$ or the median (§4).

### Is the "proportional hazards" assumption reasonable?
Cox assumes the hazard ratio is **constant over time**. Two simple checks: (1) $\ln(-\ln\hat S(t))$ curves for the groups should be roughly **parallel** (their vertical gap is $\beta$); (2) estimate the HR separately in early and late follow-up and see whether it moves.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for plan, k in km_by.items():
    k = k[(k["S"] > 0) & (k["S"] < 1)]
    ax.step(np.log(k["t"]), np.log(-np.log(k["S"])), where="post", label=plan)
ax.set_xlabel("log(time)"); ax.set_ylabel("log(-log S(t))"); ax.set_title("Roughly parallel curves support proportional hazards"); ax.legend()
plt.show()

for lo, hi in ((0, 6), (6, 30)):
    d = data[data["tenure_months"] > lo].copy()
    d["time"] = np.minimum(d["tenure_months"], hi) - lo
    d["event"] = ((d["churned"] == 1) & (d["tenure_months"] <= hi)).astype(int)
    m = PHReg.from_formula("time ~ basic", d, status=d["event"].to_numpy()).fit()
    ci_w = np.exp(m.conf_int()[0])
    print(f"months {lo:2d}-{hi:2d}: HR (basic vs premium) = {math.exp(m.params[0]):.2f}  (95% CI {ci_w[0]:.2f} to {ci_w[1]:.2f}), {int(d['event'].sum())} events")

The curves are roughly parallel, and the early (1.6) and late (2.2) hazard ratios have **broadly overlapping** intervals, so there's no clear evidence against proportional hazards. (With few events, the check has limited power: absence of evidence isn't proof.)
If hazards clearly *cross* or the ratio drifts, options include stratifying on the variable, interacting it with time, or reporting **restricted mean survival time** (RMST), the area under the KM curve up to a horizon.

## 7. Pitfalls and good practice

* **Censoring must be non-informative:** the reason someone is censored must not be related to their risk. If customers who are about to leave stop being tracked, every method here is biased.
* **Sample size = number of events**, not customers: 302 churn events support a few covariates; a rule of thumb is at least 10 events per predictor.
* **Immortal-time bias:** don't define a group by something that can only happen *after* start of follow-up (e.g. "customers who upgraded") without handling time-varying exposure.
* **Competing risks:** if people can leave for another reason (e.g. death, moving away), KM overestimates the event probability; use cumulative-incidence methods.
* **Time scale and origin** (time since sign-up, age, calendar time) must be defined clearly and the **time unit** stated.
* Report the **number at risk** under the curve and the **follow-up**; show **confidence bands**; avoid reading the far right tail.

### Reporting template
> *Time to churn was analysed from sign-up using Kaplan–Meier curves (Greenwood log-log 95% CIs) and Cox proportional-hazards regression; customers still active at the end of follow-up (24.5%) were right-censored. Median time to churn was 7.8 months (basic) and 14.5 months (premium); log-rank χ²(1) = 28.7, p < .001.
> Adjusting for monthly spend, basic-plan customers had 1.90 times the hazard of churn (95% CI 1.50 to 2.42). The proportional-hazards assumption was checked with log-minus-log plots and by estimating the hazard ratio in early and late follow-up (1.6 vs 2.2; overlapping intervals).*

## Recap

* **Censoring** means "at least this long". Ignoring it (averages, yes/no models) biases results. Use methods built for time-to-event data.
* **Kaplan–Meier** $\hat S(t)=\prod(1-d_i/n_i)$ with Greenwood intervals; **log-rank** compares curves; the **exponential** model is events ÷ total time.
* **Cox regression** gives **hazard ratios** from the partial likelihood without assuming a baseline shape; check proportional hazards.
* Report medians or $S(t)$ at a horizon *with* the hazard ratio, the numbers at risk, and the censoring assumptions.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** in the README.